In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Archimedes_Anonymization/xlmr_phi_cv_results")
print(ROOT)
print([p.name for p in ROOT.iterdir()])

/content/drive/MyDrive/Archimedes_Anonymization/xlmr_phi_cv_results
['orig_orig_orig', 'orig_syn_syn', 'syn_orig_orig', 'both_syn_syn', 'syn_syn_syn', 'both_orig_orig']


για 1

In [ ]:
# @title
import json
import pandas as pd
from pathlib import Path

exp_dir = ROOT / "both_orig_orig"

rows = []

for metrics_file in sorted(exp_dir.glob("fold_*_metrics.json")):
    with open(metrics_file, "r", encoding="utf-8") as f:
        m = json.load(f)

    row = {
        "experiment": exp_dir.name,
        "fold": m["fold"],
        "best_metric": m.get("best_metric"),
        "test_precision_phi": m.get("test_precision_phi"),
        "test_recall_phi": m.get("test_recall_phi"),
        "test_f1_phi": m.get("test_f1_phi"),
        "n_test_docs": m.get("n_test_docs"),
        "test_docs": str(m.get("test_docs"))
    }
    rows.append(row)

df_one = pd.DataFrame(rows).sort_values("fold").reset_index(drop=True)
df_one

In [ ]:
# @title
df_one[["test_precision_phi", "test_recall_phi", "test_f1_phi"]].mean()

In [ ]:
# @title
summary_one = pd.DataFrame({
    "metric": ["precision", "recall", "f1"],
    "mean": [
        df_one["test_precision_phi"].mean(),
        df_one["test_recall_phi"].mean(),
        df_one["test_f1_phi"].mean()
    ],
    "std": [
        df_one["test_precision_phi"].std(),
        df_one["test_recall_phi"].std(),
        df_one["test_f1_phi"].std()
    ]
})

summary_one

ολα

In [ ]:
# @title
import json
import pandas as pd
from pathlib import Path

all_rows = []

for exp_dir in sorted(ROOT.iterdir()):
    if not exp_dir.is_dir():
        continue

    for metrics_file in sorted(exp_dir.glob("fold_*_metrics.json")):
        with open(metrics_file, "r", encoding="utf-8") as f:
            m = json.load(f)

        row = {
            "experiment": exp_dir.name,
            "fold": m["fold"],
            "best_metric": m.get("best_metric"),
            "test_precision_phi": m.get("test_precision_phi"),
            "test_recall_phi": m.get("test_recall_phi"),
            "test_f1_phi": m.get("test_f1_phi"),
            "n_test_docs": m.get("n_test_docs"),
            "test_docs": str(m.get("test_docs"))
        }
        all_rows.append(row)

df_metrics = pd.DataFrame(all_rows).sort_values(["experiment", "fold"]).reset_index(drop=True)
df_metrics.head(10)

,experiment,fold,best_metric,test_precision_phi,test_recall_phi,test_f1_phi,n_test_docs,test_docs
0,both_orig_orig,0,0.989362,1.000000,0.920455,0.958580,5,"[0, 1, 8, 13, 22]"
1,both_orig_orig,1,0.924528,0.870588,0.913580,0.891566,5,"[2, 16, 18, 21, 23]"
2,both_orig_orig,2,0.888889,0.937888,0.974194,0.955696,5,"[5, 10, 14, 17, 19]"
3,both_orig_orig,3,0.990991,1.000000,0.984375,0.992126,5,"[6, 9, 11, 20, 24]"
4,both_orig_orig,4,0.962406,0.943662,1.000000,0.971014,5,"[3, 4, 7, 12, 15]"
5,both_syn_syn,0,0.995465,0.893112,0.984293,0.936488,5,"[0, 1, 8, 13, 22]"
6,both_syn_syn,1,0.986207,0.971510,0.924119,0.947222,5,"[2, 16, 18, 21, 23]"
7,both_syn_syn,2,0.965779,0.991495,0.941176,0.965680,5,"[5, 10, 14, 17, 19]"
8,both_syn_syn,3,0.991640,0.993432,0.946792,0.969551,5,"[6, 9, 11, 20, 24]"
9,both_syn_syn,4,0.982183,0.985932,1.000000,0.992916,5,"[3, 4, 7, 12, 15]"


In [ ]:
summary = (
    df_metrics
    .groupby("experiment")[["test_precision_phi", "test_recall_phi", "test_f1_phi"]]
    .agg(["mean", "std"])
    .round(4)
)

summary

test_precision_phi         test_recall_phi         test_f1_phi  \
                             mean     std            mean     std        mean   
experiment                                                                      
both_orig_orig             0.9504  0.0536          0.9585  0.0391      0.9538   
both_syn_syn               0.9671  0.0422          0.9593  0.0316      0.9624   
orig_orig_orig             0.9707  0.0437          0.9100  0.0590      0.9377   
orig_syn_syn               0.8937  0.1075          0.8181  0.1131      0.8530   
syn_orig_orig              0.8123  0.1232          0.8324  0.1105      0.8120   
syn_syn_syn                0.9699  0.0281          0.9807  0.0190      0.9750   

                        
                   std  
experiment              
both_orig_orig  0.0376  
both_syn_syn    0.0218  
orig_orig_orig  0.0282  
orig_syn_syn    0.1053  
syn_orig_orig   0.0651  
syn_syn_syn     0.0143

In [ ]:
def split_experiment_name(name):
    parts = name.split("_")
    return {
        "train_type": parts[0],
        "val_type": parts[1],
        "test_type": parts[2]
    }

name_parts = df_metrics["experiment"].apply(split_experiment_name).apply(pd.Series)
df_metrics = pd.concat([df_metrics, name_parts], axis=1)

df_metrics.head()

,experiment,fold,best_metric,test_precision_phi,test_recall_phi,test_f1_phi,n_test_docs,test_docs,train_type,val_type,test_type
0,both_orig_orig,0,0.989362,1.000000,0.920455,0.958580,5,"[0, 1, 8, 13, 22]",both,orig,orig
1,both_orig_orig,1,0.924528,0.870588,0.913580,0.891566,5,"[2, 16, 18, 21, 23]",both,orig,orig
2,both_orig_orig,2,0.888889,0.937888,0.974194,0.955696,5,"[5, 10, 14, 17, 19]",both,orig,orig
3,both_orig_orig,3,0.990991,1.000000,0.984375,0.992126,5,"[6, 9, 11, 20, 24]",both,orig,orig
4,both_orig_orig,4,0.962406,0.943662,1.000000,0.971014,5,"[3, 4, 7, 12, 15]",both,orig,orig


In [ ]:
for test_type in sorted(df_metrics["test_type"].unique()):
    print(f"\n===== TEST = {test_type} =====")

    display(
        df_metrics[df_metrics["test_type"] == test_type]
        .groupby("experiment")[["test_precision_phi", "test_recall_phi", "test_f1_phi"]]
        .mean()
        .round(4)
        .sort_values("test_f1_phi", ascending=False)
    )


===== TEST = orig =====


,test_precision_phi,test_recall_phi,test_f1_phi
experiment,,,
both_orig_orig,0.9504,0.9585,0.9538
orig_orig_orig,0.9707,0.9100,0.9377
syn_orig_orig,0.8123,0.8324,0.8120



===== TEST = syn =====


,test_precision_phi,test_recall_phi,test_f1_phi
experiment,,,
syn_syn_syn,0.9699,0.9807,0.9750
both_syn_syn,0.9671,0.9593,0.9624
orig_syn_syn,0.8937,0.8181,0.8530


In [ ]:
comparison_table = (
    df_metrics
    .groupby(["test_type", "experiment"])[["test_precision_phi", "test_recall_phi", "test_f1_phi"]]
    .mean()
    .round(4)
    .reset_index()
    .sort_values(["test_type", "test_f1_phi"], ascending=[True, False])
)

comparison_table

,test_type,experiment,test_precision_phi,test_recall_phi,test_f1_phi
0,orig,both_orig_orig,0.9504,0.9585,0.9538
1,orig,orig_orig_orig,0.9707,0.9100,0.9377
2,orig,syn_orig_orig,0.8123,0.8324,0.8120
5,syn,syn_syn_syn,0.9699,0.9807,0.9750
3,syn,both_syn_syn,0.9671,0.9593,0.9624
4,syn,orig_syn_syn,0.8937,0.8181,0.8530


Πιθανοτητες

In [ ]:
import json
import pickle
import numpy as np
import pandas as pd
from pathlib import Path



exp_name = "both_orig_orig"
fold = 0

exp_dir = ROOT / exp_name

arrays_path = exp_dir / f"fold_{fold}_arrays.npz"
meta_path   = exp_dir / f"fold_{fold}_meta.pkl"
metrics_path = exp_dir / f"fold_{fold}_metrics.json"

arr = np.load(arrays_path, allow_pickle=True)

with open(meta_path, "rb") as f:
    meta = pickle.load(f)

with open(metrics_path, "r", encoding="utf-8") as f:
    metrics = json.load(f)

print("NPZ keys:", arr.files)
print("Meta keys:", list(meta.keys()))
print("logits shape:", arr["logits"].shape)
print("labels shape:", arr["labels"].shape)
print("doc_ids shape:", arr["doc_ids"].shape)
print("best metric:", metrics["test_f1_phi"])

NPZ keys: ['logits', 'labels', 'doc_ids', 'instance_ids', 'original_idxs']
Meta keys: ['experiment', 'offset_mapping', 'texts_per_window', 'test_instance_texts', 'test_instance_entities', 'instance_to_original', 'instance_to_source', 'best_checkpoint', 'best_metric']
logits shape: (16, 512, 3)
labels shape: (16, 512)
doc_ids shape: (16,)
best metric: 0.9585798816568047


In [ ]:
logits = arr["logits"]
labels = arr["labels"]
doc_ids = arr["doc_ids"]

preds = logits.argmax(axis=-1)

print("preds shape:", preds.shape)
print("unique predicted labels:", np.unique(preds))
print("unique gold labels:", np.unique(labels))

preds shape: (16, 512)
unique predicted labels: [0 1 2]
unique gold labels: [-100 0 1 2]


In [ ]:
mask = labels != -100

gold_valid = labels[mask]
pred_valid = preds[mask]

print("valid tokens:", len(gold_valid))

df_token_summary = pd.DataFrame({
    "gold": gold_valid,
    "pred": pred_valid
})

df_token_summary.head()

valid tokens: 7019


,gold,pred
0,0,0
1,0,0
2,0,0
3,0,0
4,0,0


In [ ]:
pd.crosstab(
    df_token_summary["gold"],
    df_token_summary["pred"],
    rownames=["gold"],
    colnames=["pred"],
    margins=True
)

pred,0,1,2,All
gold,,,,
0,6931,0,0,6931
1,1,12,0,13
2,6,0,69,75
All,6938,12,69,7019


In [ ]:
df_windows = pd.DataFrame({
    "window_id": range(len(doc_ids)),
    "doc_id": doc_ids
})

df_windows.groupby("doc_id").size().reset_index(name="n_windows")

,doc_id,n_windows
0,0,3
1,1,4
2,8,3
3,13,3
4,22,3


In [ ]:
offset_mapping = meta["offset_mapping"]
texts_per_window = meta["texts_per_window"]

rows = []

for i in range(len(doc_ids)):
    valid_mask = labels[i] != -100

    rows.append({
        "window_id": i,
        "doc_id": int(doc_ids[i]),
        "n_valid_tokens": int(valid_mask.sum()),
        "n_errors": int((preds[i][valid_mask] != labels[i][valid_mask]).sum()),
        "window_text_preview": texts_per_window[i][:200]
    })

df_windows_detail = pd.DataFrame(rows)
df_windows_detail.head()

,window_id,doc_id,n_valid_tokens,n_errors,window_text_preview
0,0,0,510,0,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ – ΠΟΡΕΙΑ ΝΟΣΟΥ\nΑτομι...
1,1,0,510,0,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ – ΠΟΡΕΙΑ ΝΟΣΟΥ\nΑτομι...
2,2,0,425,0,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ – ΠΟΡΕΙΑ ΝΟΣΟΥ\nΑτομι...
3,3,1,510,0,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\n\nΠρογραμματισμένη ε...
4,4,1,510,0,ΑΙΤΙΑ ΕΙΣΟΔΟΥ – ΙΣΤΟΡΙΚΟ\n\nΠρογραμματισμένη ε...


In [ ]:
df_docs = (
    df_windows_detail
    .groupby("doc_id")[["n_valid_tokens", "n_errors"]]
    .sum()
    .reset_index()
)

df_docs["token_error_rate"] = df_docs["n_errors"] / df_docs["n_valid_tokens"]
df_docs = df_docs.sort_values("token_error_rate", ascending=False)

df_docs

,doc_id,n_valid_tokens,n_errors,token_error_rate
3,13,1164,7,0.006014
0,0,1445,0,0.000000
1,1,1815,0,0.000000
2,8,1247,0,0.000000
4,22,1348,0,0.000000


Entity level

In [ ]:
import json
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import re
import unicodedata

In [ ]:
def load_fold_outputs(root, exp_name, fold):
    exp_dir = Path(root) / exp_name

    arr = np.load(exp_dir / f"fold_{fold}_arrays.npz", allow_pickle=True)

    with open(exp_dir / f"fold_{fold}_meta.pkl", "rb") as f:
        meta = pickle.load(f)

    with open(exp_dir / f"fold_{fold}_metrics.json", "r", encoding="utf-8") as f:
        metrics = json.load(f)

    return {
        "exp_name": exp_name,
        "fold": fold,
        "logits": arr["logits"],   # no astype εδώ
        "labels": arr["labels"],
        "doc_ids": arr["doc_ids"],
        "instance_ids": arr["instance_ids"] if "instance_ids" in arr.files else None,
        "original_idxs": arr["original_idxs"] if "original_idxs" in arr.files else None,
        "meta": meta,
        "metrics": metrics,
    }

In [ ]:
#gia ta synthetics
def normalize_test_docs(metrics):
    return [int(x) for x in metrics.get("test_docs", [])]


def get_eval_doc_ids(outputs):
    return sorted({int(d) for d in outputs["doc_ids"]})


def _lookup_with_possible_keys(raw_dict, key):
    if key in raw_dict:
        return raw_dict[key]
    if str(key) in raw_dict:
        return raw_dict[str(key)]
    return None


def normalize_items_by_doc_id(raw, eval_doc_ids, metrics_test_docs, field_name):
    # Case 1: list/tuple -> assume same order as evaluation docs
    if isinstance(raw, (list, tuple)):
        if len(raw) != len(eval_doc_ids):
            raise ValueError(
                f"{field_name}: len(raw)={len(raw)} != len(eval_doc_ids)={len(eval_doc_ids)}"
            )
        return {eval_doc_ids[i]: raw[i] for i in range(len(eval_doc_ids))}

    # Case 2: dict -> try several matching strategies
    if isinstance(raw, dict):
        # 2a. direct match on eval_doc_ids
        if all(_lookup_with_possible_keys(raw, doc_id) is not None for doc_id in eval_doc_ids):
            return {
                doc_id: _lookup_with_possible_keys(raw, doc_id)
                for doc_id in eval_doc_ids
            }

        # 2b. direct match on metrics test_docs, then remap positionally to eval_doc_ids
        if len(metrics_test_docs) == len(eval_doc_ids) and all(
            _lookup_with_possible_keys(raw, doc_id) is not None for doc_id in metrics_test_docs
        ):
            return {
                eval_doc_ids[i]: _lookup_with_possible_keys(raw, metrics_test_docs[i])
                for i in range(len(eval_doc_ids))
            }

        # 2c. fallback: if same length, use sorted keys positionally
        raw_keys = list(raw.keys())
        if len(raw_keys) == len(eval_doc_ids):
            raw_keys_sorted = sorted(raw_keys, key=lambda x: str(x))
            return {
                eval_doc_ids[i]: raw[raw_keys_sorted[i]]
                for i in range(len(eval_doc_ids))
            }

        raise KeyError(
            f"{field_name}: could not align dict keys with eval_doc_ids={eval_doc_ids} "
            f"or metrics_test_docs={metrics_test_docs}. sample_keys={list(raw.keys())[:10]}"
        )

    raise TypeError(f"{field_name}: unsupported type {type(raw)}")

In [ ]:
def ensure_numeric_2d(x):
    x = np.asarray(x)
    if x.dtype == object:
        x = np.array([np.asarray(v, dtype=np.float32) for v in x], dtype=np.float32)
    else:
        x = x.astype(np.float32, copy=False)
    return x

In [ ]:
def normalize_texts_by_doc_id(meta, test_docs):
    raw = meta["test_instance_texts"]

    if isinstance(raw, (list, tuple)):
        if len(raw) != len(test_docs):
            raise ValueError(
                f"len(test_instance_texts)={len(raw)} != len(test_docs)={len(test_docs)}"
            )
        return {doc_id: raw[i] for i, doc_id in enumerate(test_docs)}

    if isinstance(raw, dict):
        out = {}
        for doc_id in test_docs:
            if doc_id in raw:
                out[doc_id] = raw[doc_id]
            elif str(doc_id) in raw:
                out[doc_id] = raw[str(doc_id)]
            else:
                raise KeyError(f"doc_id={doc_id} not found in test_instance_texts")
        return out

    raise TypeError(f"Unsupported type for test_instance_texts: {type(raw)}")


def normalize_entities_by_doc_id(meta, test_docs):
    raw = meta["test_instance_entities"]

    if isinstance(raw, (list, tuple)):
        if len(raw) != len(test_docs):
            raise ValueError(
                f"len(test_instance_entities)={len(raw)} != len(test_docs)={len(test_docs)}"
            )
        return {doc_id: raw[i] for i, doc_id in enumerate(test_docs)}

    if isinstance(raw, dict):
        out = {}
        for doc_id in test_docs:
            if doc_id in raw:
                out[doc_id] = raw[doc_id]
            elif str(doc_id) in raw:
                out[doc_id] = raw[str(doc_id)]
            else:
                raise KeyError(f"doc_id={doc_id} not found in test_instance_entities")
        return out

    raise TypeError(f"Unsupported type for test_instance_entities: {type(raw)}")


def normalize_gold_entities(raw_entities, text, binary_phi=True):
    spans = []

    for ent in raw_entities:
        if isinstance(ent, dict):
            start = ent.get("start", ent.get("char_start", ent.get("begin")))
            end = ent.get("end", ent.get("char_end", ent.get("stop")))
            raw_label = ent.get("label", ent.get("tag", ent.get("type", "PHI")))
            ent_text = ent.get("text", text[start:end] if start is not None and end is not None else "")

        elif isinstance(ent, (list, tuple)):
            if len(ent) < 2:
                continue
            start = ent[0]
            end = ent[1]
            raw_label = ent[2] if len(ent) >= 3 else "PHI"
            ent_text = text[start:end]

        else:
            continue

        if start is None or end is None:
            continue

        label = "PHI" if binary_phi else str(raw_label)

        spans.append({
            "start": int(start),
            "end": int(end),
            "label": label,
            "text": ent_text
        })

    return sorted(spans, key=lambda x: (x["start"], x["end"], x["label"]))

In [ ]:
O_ID = 0
B_ID = 1
I_ID = 2


def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def normalize_offsets(offsets):
    clean = []
    for off in offsets:
        if off is None:
            clean.append(None)
        elif isinstance(off, np.ndarray):
            vals = off.tolist()
            clean.append((vals[0], vals[1]) if len(vals) >= 2 else None)
        elif isinstance(off, (list, tuple)) and len(off) >= 2:
            clean.append((off[0], off[1]))
        else:
            clean.append(None)
    return clean


def bio_tokens_to_local_spans(pred_ids, offsets, probs=None, positive_label="PHI"):
    spans = []

    n = min(len(pred_ids), len(offsets))
    if probs is not None:
        n = min(n, len(probs))

    i = 0
    while i < n:
        off = offsets[i]

        if off is None or not isinstance(off, (list, tuple)) or len(off) < 2:
            i += 1
            continue

        start, end = off
        if start is None or end is None or start == end:
            i += 1
            continue

        label_id = int(pred_ids[i])

        if label_id != B_ID:
            i += 1
            continue

        span_start = start
        span_end = end
        token_probs = []

        if probs is not None:
            token_probs.append(float(probs[i, label_id]))

        j = i + 1
        while j < n:
            off_j = offsets[j]

            if off_j is None or not isinstance(off_j, (list, tuple)) or len(off_j) < 2:
                break

            s_j, e_j = off_j
            if s_j is None or e_j is None:
                break

            if s_j == e_j:
                j += 1
                continue

            if int(pred_ids[j]) == I_ID:
                span_end = e_j
                if probs is not None:
                    token_probs.append(float(probs[j, I_ID]))
                j += 1
            else:
                break

        spans.append({
            "start": int(span_start),
            "end": int(span_end),
            "label": positive_label,
            "score": float(np.mean(token_probs)) if token_probs else None
        })

        i = j

    return spans


def locate_window_in_doc(full_text, window_text, search_from=0):
    pos = full_text.find(window_text, search_from)
    if pos == -1:
        pos = full_text.find(window_text)
    return pos


def deduplicate_spans(spans):
    seen = set()
    out = []

    for sp in sorted(spans, key=lambda x: (x["start"], x["end"], x["label"])):
        key = (sp["start"], sp["end"], sp["label"])
        if key not in seen:
            seen.add(key)
            out.append(sp)

    return out


def get_doc_window_indices(outputs, doc_id):
    return [i for i, d in enumerate(outputs["doc_ids"]) if int(d) == int(doc_id)]


def get_doc_predicted_spans(outputs, doc_id, deduplicate=True, positive_label="PHI"):
    meta = outputs["meta"]
    logits_all = outputs["logits"]

    text_by_doc_id = outputs["norm"]["text_by_doc_id"]
    full_text = text_by_doc_id[int(doc_id)]

    offset_mapping = meta["offset_mapping"]
    texts_per_window = meta["texts_per_window"]

    abs_spans = []
    search_from = 0

    for widx in get_doc_window_indices(outputs, doc_id):
        window_text = texts_per_window[widx]
        local_offsets = normalize_offsets(offset_mapping[widx])

        local_logits = ensure_numeric_2d(logits_all[widx])
        local_preds = local_logits.argmax(axis=-1)
        local_probs = softmax(local_logits, axis=-1)

        window_start = locate_window_in_doc(full_text, window_text, search_from=search_from)
        if window_start == -1:
            continue

        search_from = max(search_from, window_start)

        local_spans = bio_tokens_to_local_spans(
            pred_ids=local_preds,
            offsets=local_offsets,
            probs=local_probs,
            positive_label=positive_label
        )

        for sp in local_spans:
            abs_start = window_start + sp["start"]
            abs_end = window_start + sp["end"]

            if 0 <= abs_start < abs_end <= len(full_text):
                abs_spans.append({
                    "start": abs_start,
                    "end": abs_end,
                    "label": sp["label"],
                    "text": full_text[abs_start:abs_end],
                    "score": sp["score"]
                })

    if deduplicate:
        abs_spans = deduplicate_spans(abs_spans)

    return abs_spans

In [ ]:
#normalization

BOUNDARY_CHARS = set(
    list(" \t\n\r.,;:!?…'\"“”‘’«»()[]{}<>/\\|`´~*_-")
)

PERSON_PREFIX_RE = re.compile(
    r"^(κ\.\s*|κ\s+|κος\.?\s+|κα\.?\s+|κυρ\.?\s+|κύριος\s+|κυρια\s+|κυρία\s+)",
    flags=re.IGNORECASE
)

def strip_accents(s):
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", s)

def canonical_text(s):
    """
    Canonical form for exclusion rules like ΠΓΝΙ / ΓΠΝΙωαννίνων.
    Lowercase + remove accents + keep only letters/digits.
    """
    s = strip_accents(s.lower())
    s = re.sub(r"[^0-9a-zA-Zα-ωΑ-Ω]+", "", s)
    return s

def is_pgni_non_phi(text):
    """
    Exclude mentions that refer to the local hospital and should not count as PHI.
    You can extend this list easily if you find more variants.
    """
    c = canonical_text(text)

    excluded_exact = {
        "πγνι",
        "πγνιωαννινων",
        "γπνιωαννινων",
    }

    if c in excluded_exact:
        return True

    # optional broader variants
    if c.startswith("πγνι") and "ιωαννιν" in c:
        return True
    if c.startswith("γπνι") and "ιωαννιν" in c:
        return True

    return False

def normalize_eval_span(span, doc_text):
    """
    Normalize a span by trimming harmless boundary characters and common
    person prefixes like 'κ.' from the LEFT boundary only.
    Returns normalized span dict or None if span becomes empty / excluded.
    """
    start = int(span["start"])
    end = int(span["end"])
    label = span["label"]

    if start >= end:
        return None

    # 1) trim generic boundary chars from both ends
    while start < end and doc_text[start] in BOUNDARY_CHARS:
        start += 1
    while start < end and doc_text[end - 1] in BOUNDARY_CHARS:
        end -= 1

    if start >= end:
        return None

    # 2) repeatedly remove wrapper pairs like <<...>>, «...», "..."
    changed = True
    while changed and start < end:
        changed = False
        surface = doc_text[start:end].strip()

        pairs = [
            ("<<", ">>"),
            ("«", "»"),
            ('"', '"'),
            ("“", "”"),
            ("‘", "’"),
            ("(", ")"),
            ("[", "]"),
            ("{", "}"),
        ]

        for left, right in pairs:
            if surface.startswith(left) and surface.endswith(right) and len(surface) > len(left) + len(right):
                # find actual local boundaries inside current slice
                raw = doc_text[start:end]
                lpos = raw.find(left)
                rpos = raw.rfind(right)
                if lpos != -1 and rpos != -1 and (rpos + len(right)) <= len(raw):
                    start += lpos + len(left)
                    end = start + (rpos - (lpos + len(left)))
                    changed = True
                    break

        while start < end and doc_text[start] in BOUNDARY_CHARS:
            start += 1
        while start < end and doc_text[end - 1] in BOUNDARY_CHARS:
            end -= 1

    if start >= end:
        return None

    # 3) remove common prefixes like "κ. ", "κος ", "κα ", etc.
    #    only from the LEFT boundary
    surface = doc_text[start:end]
    m = PERSON_PREFIX_RE.match(surface)
    if m:
        start += len(m.group(0))

    while start < end and doc_text[start] in BOUNDARY_CHARS:
        start += 1
    while start < end and doc_text[end - 1] in BOUNDARY_CHARS:
        end -= 1

    if start >= end:
        return None

    norm_text = doc_text[start:end]

    # 4) exclude non-PHI hospital mentions
    if is_pgni_non_phi(norm_text):
        return None

    return {
        "start": int(start),
        "end": int(end),
        "label": label,
        "text": norm_text,
        "score": span.get("score")
    }

def normalize_spans_for_eval(spans, doc_text):
    out = []
    seen = set()

    for sp in spans:
        norm = normalize_eval_span(sp, doc_text)
        if norm is None:
            continue

        key = (norm["start"], norm["end"], norm["label"])
        if key not in seen:
            seen.add(key)
            out.append(norm)

    return sorted(out, key=lambda x: (x["start"], x["end"], x["label"]))

In [ ]:
def span_key(span):
    return (span["start"], span["end"], span["label"])


def spans_to_set(spans):
    return {span_key(s) for s in spans}


def overlaps(a, b):
    return not (a["end"] <= b["start"] or b["end"] <= a["start"])


def same_label(a, b):
    return a["label"] == b["label"]


def greedy_partial_match(pred_spans, gold_spans, require_same_label=True):
    matched_pred = set()
    matched_gold = set()
    matches = []

    for i, p in enumerate(pred_spans):
        for j, g in enumerate(gold_spans):
            if j in matched_gold:
                continue

            if require_same_label and not same_label(p, g):
                continue

            if overlaps(p, g):
                matched_pred.add(i)
                matched_gold.add(j)
                matches.append((i, j))
                break

    return matched_pred, matched_gold, matches


def compute_entity_metrics_for_doc(pred_spans, gold_spans, mode="strict"):
    pred_spans = sorted(pred_spans, key=lambda x: (x["start"], x["end"], x["label"]))
    gold_spans = sorted(gold_spans, key=lambda x: (x["start"], x["end"], x["label"]))

    if mode == "strict":
        pred_set = spans_to_set(pred_spans)
        gold_set = spans_to_set(gold_spans)

        tp = len(pred_set & gold_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)

    elif mode == "partial":
        matched_pred, matched_gold, _ = greedy_partial_match(
            pred_spans, gold_spans, require_same_label=True
        )
        tp = len(matched_pred)
        fp = len(pred_spans) - tp
        fn = len(gold_spans) - tp

    else:
        raise ValueError("mode must be 'strict' or 'partial'")

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def aggregate_entity_metrics(per_doc_metrics):
    tp = sum(x["tp"] for x in per_doc_metrics)
    fp = sum(x["fp"] for x in per_doc_metrics)
    fn = sum(x["fn"] for x in per_doc_metrics)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

In [ ]:
def prepare_outputs(root, exp_name, fold):
    outputs = load_fold_outputs(root, exp_name, fold)

    metrics_test_docs = normalize_test_docs(outputs["metrics"])
    eval_doc_ids = get_eval_doc_ids(outputs)

    text_by_doc_id = normalize_items_by_doc_id(
        raw=outputs["meta"]["test_instance_texts"],
        eval_doc_ids=eval_doc_ids,
        metrics_test_docs=metrics_test_docs,
        field_name="test_instance_texts",
    )

    entities_by_doc_id = normalize_items_by_doc_id(
        raw=outputs["meta"]["test_instance_entities"],
        eval_doc_ids=eval_doc_ids,
        metrics_test_docs=metrics_test_docs,
        field_name="test_instance_entities",
    )

    outputs["norm"] = {
        "metrics_test_docs": metrics_test_docs,
        "eval_doc_ids": eval_doc_ids,
        "text_by_doc_id": text_by_doc_id,
        "entities_by_doc_id": entities_by_doc_id,
    }

    return outputs

def evaluate_entity_level_fold(root, exp_name, fold, positive_label="PHI", deduplicate=True):
    outputs = prepare_outputs(root, exp_name, fold)

    strict_doc_metrics = []
    partial_doc_metrics = []

    for doc_id in outputs["norm"]["eval_doc_ids"]:
        text = outputs["norm"]["text_by_doc_id"][doc_id]
        gold_raw = outputs["norm"]["entities_by_doc_id"][doc_id]

        # raw spans
        gold_spans_raw = normalize_gold_entities(gold_raw, text, binary_phi=True)
        pred_spans_raw = get_doc_predicted_spans(
            outputs,
            doc_id=doc_id,
            deduplicate=deduplicate,
            positive_label=positive_label
        )

        # normalized spans for evaluation
        gold_spans = normalize_spans_for_eval(gold_spans_raw, text)
        pred_spans = normalize_spans_for_eval(pred_spans_raw, text)

        strict_doc_metrics.append(
            compute_entity_metrics_for_doc(pred_spans, gold_spans, mode="strict")
        )
        partial_doc_metrics.append(
            compute_entity_metrics_for_doc(pred_spans, gold_spans, mode="partial")
        )

    strict = aggregate_entity_metrics(strict_doc_metrics)
    partial = aggregate_entity_metrics(partial_doc_metrics)
    m = outputs["metrics"]

    return {
        "experiment": exp_name,
        "fold": fold,

        "token_precision_phi": m.get("test_precision_phi"),
        "token_recall_phi": m.get("test_recall_phi"),
        "token_f1_phi": m.get("test_f1_phi"),

        "entity_strict_precision": strict["precision"],
        "entity_strict_recall": strict["recall"],
        "entity_strict_f1": strict["f1"],
        "entity_strict_tp": strict["tp"],
        "entity_strict_fp": strict["fp"],
        "entity_strict_fn": strict["fn"],

        "entity_partial_precision": partial["precision"],
        "entity_partial_recall": partial["recall"],
        "entity_partial_f1": partial["f1"],
        "entity_partial_tp": partial["tp"],
        "entity_partial_fp": partial["fp"],
        "entity_partial_fn": partial["fn"],

        "n_test_docs": m.get("n_test_docs"),
        "test_docs": str(m.get("test_docs")),
        "eval_doc_ids": str(outputs["norm"]["eval_doc_ids"]),
    }

In [ ]:
def list_experiments(root):
    root = Path(root)
    return sorted([p.name for p in root.iterdir() if p.is_dir()])


def list_folds_for_experiment(root, exp_name):
    exp_dir = Path(root) / exp_name
    folds = []

    for p in exp_dir.glob("fold_*_metrics.json"):
        name = p.stem  # fold_0_metrics
        fold = int(name.split("_")[1])
        folds.append(fold)

    return sorted(set(folds))


def build_entity_results_df(root, positive_label="PHI", deduplicate=True):
    rows = []

    for exp_name in list_experiments(root):
        folds = list_folds_for_experiment(root, exp_name)

        for fold in folds:
            try:
                row = evaluate_entity_level_fold(
                    root=root,
                    exp_name=exp_name,
                    fold=fold,
                    positive_label=positive_label,
                    deduplicate=deduplicate
                )
                rows.append(row)
            except Exception as e:
                rows.append({
                    "experiment": exp_name,
                    "fold": fold,
                    "error": str(e)
                })

    return pd.DataFrame(rows).sort_values(["experiment", "fold"]).reset_index(drop=True)

In [ ]:
entity_results_df = build_entity_results_df(ROOT)
entity_results_df.head(10)

,experiment,fold,token_precision_phi,token_recall_phi,token_f1_phi,entity_strict_precision,entity_strict_recall,entity_strict_f1,entity_strict_tp,entity_strict_fp,...,entity_partial_precision,entity_partial_recall,entity_partial_f1,entity_partial_tp,entity_partial_fp,entity_partial_fn,n_test_docs,test_docs,eval_doc_ids,error
0,both_orig_orig,0,1.000000,0.920455,0.958580,0.727273,0.727273,0.727273,8.0,3.0,...,1.000000,1.000000,1.000000,11.0,0.0,0.0,5.0,"[0, 1, 8, 13, 22]","[0, 1, 8, 13, 22]",NaN
1,both_orig_orig,1,0.870588,0.913580,0.891566,0.875000,0.700000,0.777778,7.0,1.0,...,0.875000,0.700000,0.777778,7.0,1.0,3.0,5.0,"[2, 16, 18, 21, 23]","[2, 16, 18, 21, 23]",NaN
2,both_orig_orig,2,0.937888,0.974194,0.955696,1.000000,0.909091,0.952381,10.0,0.0,...,1.000000,0.909091,0.952381,10.0,0.0,1.0,5.0,"[5, 10, 14, 17, 19]","[5, 10, 14, 17, 19]",NaN
3,both_orig_orig,3,1.000000,0.984375,0.992126,1.000000,0.857143,0.923077,6.0,0.0,...,1.000000,0.857143,0.923077,6.0,0.0,1.0,5.0,"[6, 9, 11, 20, 24]","[6, 9, 11, 20, 24]",NaN
4,both_orig_orig,4,0.943662,1.000000,0.971014,1.000000,1.000000,1.000000,6.0,0.0,...,1.000000,1.000000,1.000000,6.0,0.0,0.0,5.0,"[3, 4, 7, 12, 15]","[3, 4, 7, 12, 15]",NaN
5,both_syn_syn,0,0.893112,0.984293,0.936488,0.762500,0.835616,0.797386,61.0,19.0,...,0.887500,0.972603,0.928105,71.0,9.0,2.0,5.0,"[0, 1, 8, 13, 22]","[25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 65, 6...",NaN
6,both_syn_syn,1,0.971510,0.924119,0.947222,0.923913,0.841584,0.880829,85.0,7.0,...,0.989130,0.900990,0.943005,91.0,1.0,10.0,5.0,"[2, 16, 18, 21, 23]","[35, 36, 37, 38, 39, 80, 81, 82, 83, 84, 90, 9...",NaN
7,both_syn_syn,2,0.991495,0.941176,0.965680,0.983871,0.871429,0.924242,122.0,2.0,...,0.991935,0.878571,0.931818,123.0,1.0,17.0,5.0,"[5, 10, 14, 17, 19]","[50, 51, 52, 53, 54, 105, 106, 107, 108, 109, ...",NaN
8,both_syn_syn,3,0.993432,0.946792,0.969551,0.988889,0.978022,0.983425,89.0,1.0,...,0.988889,0.978022,0.983425,89.0,1.0,2.0,5.0,"[6, 9, 11, 20, 24]","[55, 56, 57, 58, 59, 70, 71, 72, 73, 74, 75, 7...",NaN
9,both_syn_syn,4,0.985932,1.000000,0.992916,0.960396,1.000000,0.979798,97.0,4.0,...,0.960396,1.000000,0.979798,97.0,4.0,0.0,5.0,"[3, 4, 7, 12, 15]","[40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 60, 6...",NaN


In [ ]:
def split_experiment_name(name):
    parts = name.split("_")
    if len(parts) != 3:
        return {
            "train_type": None,
            "val_type": None,
            "test_type": None
        }
    return {
        "train_type": parts[0],
        "val_type": parts[1],
        "test_type": parts[2]
    }


name_parts = entity_results_df["experiment"].apply(split_experiment_name).apply(pd.Series)
entity_results_df = pd.concat([entity_results_df, name_parts], axis=1)

entity_results_df.head()

,experiment,fold,token_precision_phi,token_recall_phi,token_f1_phi,entity_strict_precision,entity_strict_recall,entity_strict_f1,entity_strict_tp,entity_strict_fp,...,entity_partial_tp,entity_partial_fp,entity_partial_fn,n_test_docs,test_docs,eval_doc_ids,error,train_type,val_type,test_type
0,both_orig_orig,0,1.000000,0.920455,0.958580,0.727273,0.727273,0.727273,8.0,3.0,...,11.0,0.0,0.0,5.0,"[0, 1, 8, 13, 22]","[0, 1, 8, 13, 22]",NaN,both,orig,orig
1,both_orig_orig,1,0.870588,0.913580,0.891566,0.875000,0.700000,0.777778,7.0,1.0,...,7.0,1.0,3.0,5.0,"[2, 16, 18, 21, 23]","[2, 16, 18, 21, 23]",NaN,both,orig,orig
2,both_orig_orig,2,0.937888,0.974194,0.955696,1.000000,0.909091,0.952381,10.0,0.0,...,10.0,0.0,1.0,5.0,"[5, 10, 14, 17, 19]","[5, 10, 14, 17, 19]",NaN,both,orig,orig
3,both_orig_orig,3,1.000000,0.984375,0.992126,1.000000,0.857143,0.923077,6.0,0.0,...,6.0,0.0,1.0,5.0,"[6, 9, 11, 20, 24]","[6, 9, 11, 20, 24]",NaN,both,orig,orig
4,both_orig_orig,4,0.943662,1.000000,0.971014,1.000000,1.000000,1.000000,6.0,0.0,...,6.0,0.0,0.0,5.0,"[3, 4, 7, 12, 15]","[3, 4, 7, 12, 15]",NaN,both,orig,orig


In [ ]:
summary_entity = (
    entity_results_df
    .groupby("experiment")[[
        "token_f1_phi",
        "entity_strict_f1",
        "entity_partial_f1"
    ]]
    .agg(["mean", "std"])
    .round(4)
)

summary_entity

token_f1_phi         entity_strict_f1          \
                       mean     std             mean     std   
experiment                                                     
both_orig_orig       0.9538  0.0376           0.8761  0.1175   
both_syn_syn         0.9624  0.0218           0.9131  0.0774   
orig_orig_orig       0.9377  0.0282           0.6937  0.2747   
orig_syn_syn         0.8979  0.0366           0.7070  0.1302   
syn_orig_orig        0.8120  0.0651           0.5371  0.2903   
syn_syn_syn          0.9750  0.0143           0.9299  0.0670   

               entity_partial_f1          
                            mean     std  
experiment                                
both_orig_orig            0.9306  0.0915  
both_syn_syn              0.9532  0.0265  
orig_orig_orig            0.6937  0.2747  
orig_syn_syn              0.7221  0.1275  
syn_orig_orig             0.7938  0.1415  
syn_syn_syn               0.9706  0.0105

In [ ]:
for test_type in sorted(entity_results_df["test_type"].dropna().unique()):
    print(f"\n===== TEST = {test_type} =====")

    display(
        entity_results_df[entity_results_df["test_type"] == test_type]
        .groupby("experiment")[[
            "token_recall_phi",
            "entity_strict_recall",
            "entity_partial_recall"
        ]]
        .mean()
        .round(4)
        .sort_values("entity_strict_recall", ascending=False)
    )


===== TEST = orig =====


,token_recall_phi,entity_strict_recall,entity_partial_recall
experiment,,,
both_orig_orig,0.9585,0.8387,0.8932
syn_orig_orig,0.8324,0.6206,0.8758
orig_orig_orig,0.9100,0.5835,0.5835



===== TEST = syn =====


,token_recall_phi,entity_strict_recall,entity_partial_recall
experiment,,,
syn_syn_syn,0.9807,0.9344,0.9759
both_syn_syn,0.9593,0.9053,0.9460
orig_syn_syn,0.8682,0.5675,0.5796


In [ ]:
for test_type in sorted(entity_results_df["test_type"].dropna().unique()):
    print(f"\n===== TEST = {test_type} =====")

    display(
        entity_results_df[entity_results_df["test_type"] == test_type]
        .groupby("experiment")[[
            "token_precision_phi",
            "entity_strict_precision",
            "entity_partial_precision"
        ]]
        .mean()
        .round(4)
        .sort_values("entity_strict_precision", ascending=False)
    )


===== TEST = orig =====


,token_precision_phi,entity_strict_precision,entity_partial_precision
experiment,,,
orig_orig_orig,0.9707,1.0000,1.0000
both_orig_orig,0.9504,0.9205,0.9750
syn_orig_orig,0.8123,0.4883,0.7775



===== TEST = syn =====


,token_precision_phi,entity_strict_precision,entity_partial_precision
experiment,,,
orig_syn_syn,0.9322,0.9638,0.9854
syn_syn_syn,0.9699,0.9258,0.9657
both_syn_syn,0.9671,0.9239,0.9636


In [ ]:
for test_type in sorted(entity_results_df["test_type"].dropna().unique()):
    print(f"\n===== TEST = {test_type} =====")

    display(
        entity_results_df[entity_results_df["test_type"] == test_type]
        .groupby("experiment")[[
            "token_f1_phi",
            "entity_strict_f1",
            "entity_partial_f1"
        ]]
        .mean()
        .round(4)
        .sort_values("entity_strict_f1", ascending=False)
    )


===== TEST = orig =====


,token_f1_phi,entity_strict_f1,entity_partial_f1
experiment,,,
both_orig_orig,0.9538,0.6986,0.8987
orig_orig_orig,0.9377,0.6029,0.6773
syn_orig_orig,0.8120,0.3315,0.7167



===== TEST = syn =====


,token_f1_phi,entity_strict_f1,entity_partial_f1
experiment,,,
syn_syn_syn,0.9750,0.9247,0.9674
both_syn_syn,0.9624,0.8875,0.9523
orig_syn_syn,0.8979,0.6892,0.7202


In [ ]:
strict_df = entity_results_df[[
    "experiment",
    "fold",
    "entity_strict_precision",
    "entity_strict_recall",
    "entity_strict_f1"
]].copy()

strict_df

,experiment,fold,entity_strict_precision,entity_strict_recall,entity_strict_f1
0,both_orig_orig,0,0.727273,0.727273,0.727273
1,both_orig_orig,1,0.875000,0.700000,0.777778
2,both_orig_orig,2,1.000000,0.909091,0.952381
3,both_orig_orig,3,1.000000,0.857143,0.923077
4,both_orig_orig,4,1.000000,1.000000,1.000000
5,both_syn_syn,0,0.762500,0.835616,0.797386
6,both_syn_syn,1,0.923913,0.841584,0.880829
7,both_syn_syn,2,0.983871,0.871429,0.924242
8,both_syn_syn,3,0.988889,0.978022,0.983425
9,both_syn_syn,4,0.960396,1.000000,0.979798


In [ ]:
exp_strict_summary = (
    entity_results_df.groupby("experiment", as_index=False)
      .agg(
          strict_precision_mean=("entity_strict_precision", "mean"),
          strict_recall_mean=("entity_strict_recall", "mean"),
          strict_f1_mean=("entity_strict_f1", "mean"),
          n_folds=("fold", "count")
      )
      .sort_values("strict_f1_mean", ascending=False)
      .reset_index(drop=True)
)

exp_strict_summary

,experiment,strict_precision_mean,strict_recall_mean,strict_f1_mean,n_folds
0,syn_syn_syn,0.925810,0.934389,0.929877,5
1,both_syn_syn,0.923914,0.905330,0.913136,5
2,both_orig_orig,0.920455,0.838701,0.876102,5
3,orig_syn_syn,0.963817,0.567540,0.706963,5
4,orig_orig_orig,1.000000,0.583463,0.693651,5
5,syn_orig_orig,0.488333,0.620606,0.537124,5


In [ ]:
def inspect_experiment_predictions(
    root,
    exp_name,
    fold,
    positive_label="PHI",
    deduplicate=True,
    max_docs=None,
    return_results=False,
    only_errors=False
):
    outputs = prepare_outputs(root, exp_name, fold)

    results = []

    for i, doc_id in enumerate(outputs["norm"]["eval_doc_ids"]):
        if max_docs is not None and i >= max_docs:
            break

        text = outputs["norm"]["text_by_doc_id"][doc_id]
        gold_raw = outputs["norm"]["entities_by_doc_id"][doc_id]

        gold_spans_raw = normalize_gold_entities(gold_raw, text, binary_phi=True)
        pred_spans_raw = get_doc_predicted_spans(
            outputs,
            doc_id=doc_id,
            deduplicate=deduplicate,
            positive_label=positive_label
        )

        gold_spans = normalize_spans_for_eval(gold_spans_raw, text)
        pred_spans = normalize_spans_for_eval(pred_spans_raw, text)

        gold_keys = {(g["start"], g["end"], g["text"]) for g in gold_spans}
        pred_keys = {(p["start"], p["end"], p["text"]) for p in pred_spans}

        if only_errors:
            gold_out = [
                {"start": g["start"], "end": g["end"], "text": g["text"]}
                for g in gold_spans
                if (g["start"], g["end"], g["text"]) not in pred_keys
            ]
            pred_out = [
                {"start": p["start"], "end": p["end"], "text": p["text"], "score": p.get("score")}
                for p in pred_spans
                if (p["start"], p["end"], p["text"]) not in gold_keys
            ]
        else:
            gold_out = [
                {"start": g["start"], "end": g["end"], "text": g["text"]}
                for g in gold_spans
            ]
            pred_out = [
                {"start": p["start"], "end": p["end"], "text": p["text"], "score": p.get("score")}
                for p in pred_spans
            ]

        if only_errors and not gold_out and not pred_out:
            continue

        results.append({
            "doc_id": doc_id,
            "gold": gold_out,
            "pred": pred_out,
        })

        print("\n" + "=" * 70)
        print(f"EXP: {exp_name} | FOLD: {fold} | DOC: {doc_id}")
        print("=" * 70)

        print("\nGOLD:")
        for g in gold_out:
            print(g)

        print("\nPRED:")
        for p in pred_out:
            print(p)

    if return_results:
        return results
    return None

In [ ]:
def inspect_experiment_all_folds(
    root,
    exp_name,
    positive_label="PHI",
    deduplicate=True,
    max_docs_per_fold=None,
    return_results=False,
    only_errors=False
):
    folds = list_folds_for_experiment(root, exp_name)

    all_results = {}

    for fold in folds:
        print("\n" + "#" * 80)
        print(f"EXPERIMENT: {exp_name} | FOLD: {fold}")
        print("#" * 80)

        fold_results = inspect_experiment_predictions(
            root=root,
            exp_name=exp_name,
            fold=fold,
            positive_label=positive_label,
            deduplicate=deduplicate,
            max_docs=max_docs_per_fold,
            return_results=return_results,
            only_errors=only_errors
        )

        if return_results:
            all_results[fold] = fold_results

    if return_results:
        return all_results
    return None

In [ ]:
inspect_experiment_all_folds(
    root=ROOT,
    exp_name="syn_orig_orig",
    max_docs_per_fold=None,
    only_errors=True,
    return_results=False
)


################################################################################
EXPERIMENT: syn_orig_orig | FOLD: 0
################################################################################

EXP: syn_orig_orig | FOLD: 0 | DOC: 0

GOLD:
{'start': 298, 'end': 320, 'text': 'ΓΝ Αθηνών «Ιπποκράτειο'}
{'start': 366, 'end': 388, 'text': 'ΓΝ Αθηνών «Ιπποκράτειο'}
{'start': 634, 'end': 656, 'text': 'ΓΝ Αθηνών «Ιπποκράτειο'}

PRED:
{'start': 298, 'end': 307, 'text': 'ΓΝ Αθηνών', 'score': 0.9822265903155009}
{'start': 309, 'end': 320, 'text': 'Ιπποκράτειο', 'score': 0.9944399297237396}
{'start': 366, 'end': 375, 'text': 'ΓΝ Αθηνών', 'score': 0.9828918774922689}
{'start': 377, 'end': 388, 'text': 'Ιπποκράτειο', 'score': 0.9943097134431204}
{'start': 634, 'end': 643, 'text': 'ΓΝ Αθηνών', 'score': 0.9822951555252075}
{'start': 645, 'end': 656, 'text': 'Ιπποκράτειο', 'score': 0.994359294573466}

EXP: syn_orig_orig | FOLD: 0 | DOC: 1

GOLD:
{'start': 60, 'end': 89, 'text': 'ΓΝ Θεσσαλονίκης «Π

In [ ]:
import json
import pickle
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd


# =========================================================
# Load / normalize experiment outputs
# =========================================================

def load_fold_outputs(root, exp_name, fold):
    exp_dir = Path(root) / exp_name

    arr = np.load(exp_dir / f"fold_{fold}_arrays.npz", allow_pickle=True)

    with open(exp_dir / f"fold_{fold}_meta.pkl", "rb") as f:
        meta = pickle.load(f)

    with open(exp_dir / f"fold_{fold}_metrics.json", "r", encoding="utf-8") as f:
        metrics = json.load(f)

    return {
        "exp_name": exp_name,
        "fold": fold,
        "logits": arr["logits"],
        "labels": arr["labels"],
        "doc_ids": arr["doc_ids"],
        "instance_ids": arr["instance_ids"] if "instance_ids" in arr.files else None,
        "original_idxs": arr["original_idxs"] if "original_idxs" in arr.files else None,
        "meta": meta,
        "metrics": metrics,
    }


def normalize_test_docs(metrics):
    return [int(x) for x in metrics.get("test_docs", [])]


def get_eval_doc_ids(outputs):
    return sorted({int(d) for d in outputs["doc_ids"]})


def _lookup_with_possible_keys(raw_dict, key):
    if key in raw_dict:
        return raw_dict[key]
    if str(key) in raw_dict:
        return raw_dict[str(key)]
    return None


def normalize_items_by_doc_id(raw, eval_doc_ids, metrics_test_docs, field_name):
    if isinstance(raw, (list, tuple)):
        if len(raw) != len(eval_doc_ids):
            raise ValueError(
                f"{field_name}: len(raw)={len(raw)} != len(eval_doc_ids)={len(eval_doc_ids)}"
            )
        return {eval_doc_ids[i]: raw[i] for i in range(len(eval_doc_ids))}

    if isinstance(raw, dict):
        if all(_lookup_with_possible_keys(raw, doc_id) is not None for doc_id in eval_doc_ids):
            return {
                doc_id: _lookup_with_possible_keys(raw, doc_id)
                for doc_id in eval_doc_ids
            }

        if len(metrics_test_docs) == len(eval_doc_ids) and all(
            _lookup_with_possible_keys(raw, doc_id) is not None for doc_id in metrics_test_docs
        ):
            return {
                eval_doc_ids[i]: _lookup_with_possible_keys(raw, metrics_test_docs[i])
                for i in range(len(eval_doc_ids))
            }

        raw_keys = list(raw.keys())
        if len(raw_keys) == len(eval_doc_ids):
            raw_keys_sorted = sorted(raw_keys, key=lambda x: str(x))
            return {
                eval_doc_ids[i]: raw[raw_keys_sorted[i]]
                for i in range(len(eval_doc_ids))
            }

        raise KeyError(
            f"{field_name}: could not align dict keys with eval_doc_ids={eval_doc_ids} "
            f"or metrics_test_docs={metrics_test_docs}. sample_keys={list(raw.keys())[:10]}"
        )

    raise TypeError(f"{field_name}: unsupported type {type(raw)}")


def ensure_numeric_2d(x):
    x = np.asarray(x)
    if x.dtype == object:
        x = np.array([np.asarray(v, dtype=np.float32) for v in x], dtype=np.float32)
    else:
        x = x.astype(np.float32, copy=False)
    return x


def normalize_gold_entities(raw_entities, text, binary_phi=True):
    spans = []

    for ent in raw_entities:
        if isinstance(ent, dict):
            start = ent.get("start", ent.get("char_start", ent.get("begin")))
            end = ent.get("end", ent.get("char_end", ent.get("stop")))
            raw_label = ent.get("label", ent.get("tag", ent.get("type", "PHI")))
            ent_text = ent.get("text", text[start:end] if start is not None and end is not None else "")

        elif isinstance(ent, (list, tuple)):
            if len(ent) < 2:
                continue
            start = ent[0]
            end = ent[1]
            raw_label = ent[2] if len(ent) >= 3 else "PHI"
            ent_text = text[start:end]

        else:
            continue

        if start is None or end is None:
            continue

        label = "PHI" if binary_phi else str(raw_label)

        spans.append({
            "start": int(start),
            "end": int(end),
            "label": label,
            "text": ent_text
        })

    return sorted(spans, key=lambda x: (x["start"], x["end"], x["label"]))


def prepare_outputs(root, exp_name, fold):
    outputs = load_fold_outputs(root, exp_name, fold)

    metrics_test_docs = normalize_test_docs(outputs["metrics"])
    eval_doc_ids = get_eval_doc_ids(outputs)

    text_by_doc_id = normalize_items_by_doc_id(
        raw=outputs["meta"]["test_instance_texts"],
        eval_doc_ids=eval_doc_ids,
        metrics_test_docs=metrics_test_docs,
        field_name="test_instance_texts",
    )

    entities_by_doc_id = normalize_items_by_doc_id(
        raw=outputs["meta"]["test_instance_entities"],
        eval_doc_ids=eval_doc_ids,
        metrics_test_docs=metrics_test_docs,
        field_name="test_instance_entities",
    )

    outputs["norm"] = {
        "metrics_test_docs": metrics_test_docs,
        "eval_doc_ids": eval_doc_ids,
        "text_by_doc_id": text_by_doc_id,
        "entities_by_doc_id": entities_by_doc_id,
    }

    return outputs


# =========================================================
# Token -> span
# =========================================================

O_ID = 0
B_ID = 1
I_ID = 2


def softmax(x, axis=-1):
    x = x - np.max(x, axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)


def normalize_offsets(offsets):
    clean = []
    for off in offsets:
        if off is None:
            clean.append(None)
        elif isinstance(off, np.ndarray):
            vals = off.tolist()
            clean.append((vals[0], vals[1]) if len(vals) >= 2 else None)
        elif isinstance(off, (list, tuple)) and len(off) >= 2:
            clean.append((off[0], off[1]))
        else:
            clean.append(None)
    return clean


def bio_tokens_to_local_spans(pred_ids, offsets, probs=None, positive_label="PHI"):
    spans = []

    n = min(len(pred_ids), len(offsets))
    if probs is not None:
        n = min(n, len(probs))

    i = 0
    while i < n:
        off = offsets[i]

        if off is None or not isinstance(off, (list, tuple)) or len(off) < 2:
            i += 1
            continue

        start, end = off
        if start is None or end is None or start == end:
            i += 1
            continue

        label_id = int(pred_ids[i])

        if label_id != B_ID:
            i += 1
            continue

        span_start = start
        span_end = end
        token_probs = []

        if probs is not None:
            token_probs.append(float(probs[i, label_id]))

        j = i + 1
        while j < n:
            off_j = offsets[j]

            if off_j is None or not isinstance(off_j, (list, tuple)) or len(off_j) < 2:
                break

            s_j, e_j = off_j
            if s_j is None or e_j is None:
                break

            if s_j == e_j:
                j += 1
                continue

            if int(pred_ids[j]) == I_ID:
                span_end = e_j
                if probs is not None:
                    token_probs.append(float(probs[j, I_ID]))
                j += 1
            else:
                break

        spans.append({
            "start": int(span_start),
            "end": int(span_end),
            "label": positive_label,
            "score": float(np.mean(token_probs)) if token_probs else None
        })

        i = j

    return spans


def locate_window_in_doc(full_text, window_text, search_from=0):
    pos = full_text.find(window_text, search_from)
    if pos == -1:
        pos = full_text.find(window_text)
    return pos


def deduplicate_spans(spans):
    seen = set()
    out = []

    for sp in sorted(spans, key=lambda x: (x["start"], x["end"], x["label"])):
        key = (sp["start"], sp["end"], sp["label"])
        if key not in seen:
            seen.add(key)
            out.append(sp)

    return out


def get_doc_window_indices(outputs, doc_id):
    return [i for i, d in enumerate(outputs["doc_ids"]) if int(d) == int(doc_id)]


def get_doc_predicted_spans(outputs, doc_id, deduplicate=True, positive_label="PHI"):
    meta = outputs["meta"]
    logits_all = outputs["logits"]

    text_by_doc_id = outputs["norm"]["text_by_doc_id"]
    full_text = text_by_doc_id[int(doc_id)]

    offset_mapping = meta["offset_mapping"]
    texts_per_window = meta["texts_per_window"]

    abs_spans = []
    search_from = 0

    for widx in get_doc_window_indices(outputs, doc_id):
        window_text = texts_per_window[widx]
        local_offsets = normalize_offsets(offset_mapping[widx])

        local_logits = ensure_numeric_2d(logits_all[widx])
        local_preds = local_logits.argmax(axis=-1)
        local_probs = softmax(local_logits, axis=-1)

        window_start = locate_window_in_doc(full_text, window_text, search_from=search_from)
        if window_start == -1:
            continue

        search_from = max(search_from, window_start)

        local_spans = bio_tokens_to_local_spans(
            pred_ids=local_preds,
            offsets=local_offsets,
            probs=local_probs,
            positive_label=positive_label
        )

        for sp in local_spans:
            abs_start = window_start + sp["start"]
            abs_end = window_start + sp["end"]

            if 0 <= abs_start < abs_end <= len(full_text):
                abs_spans.append({
                    "start": abs_start,
                    "end": abs_end,
                    "label": sp["label"],
                    "text": full_text[abs_start:abs_end],
                    "score": sp["score"]
                })

    if deduplicate:
        abs_spans = deduplicate_spans(abs_spans)

    return abs_spans


# =========================================================
# Normalization / exclusions / merging
# =========================================================

BOUNDARY_CHARS = set(list(" \t\n\r.,;:!?…'\"“”‘’«»()[]{}<>/\\|`´~*_-"))


PERSON_PREFIX_RE = re.compile(
    r"^(κ\.\s*|κ\s+|κος\.?\s+|κα\.?\s+|κυρ\.?\s+|κύριος\s+|κυρια\s+|κυρία\s+)",
    flags=re.IGNORECASE
)


def strip_accents(s):
    s = unicodedata.normalize("NFD", s)
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")
    return unicodedata.normalize("NFC", s)


def canonical_text(s):
    s = strip_accents(s.lower())
    s = re.sub(r"[^0-9a-zA-Zα-ωΑ-Ω]+", "", s)
    return s


def is_pgni_non_phi(text):
    c = canonical_text(text)

    excluded_exact = {
        "πγνι",
        "πγνιωαννινων",
        "γπνιωαννινων",
    }

    if c in excluded_exact:
        return True

    if c.startswith("πγνι") and "ιωαννιν" in c:
        return True
    if c.startswith("γπνι") and "ιωαννιν" in c:
        return True

    return False


def normalize_eval_span(span, doc_text):
    start = int(span["start"])
    end = int(span["end"])
    label = span["label"]

    if start >= end:
        return None

    while start < end and doc_text[start] in BOUNDARY_CHARS:
        start += 1
    while start < end and doc_text[end - 1] in BOUNDARY_CHARS:
        end -= 1

    if start >= end:
        return None

    changed = True
    while changed and start < end:
        changed = False
        surface = doc_text[start:end].strip()

        pairs = [
            ("<<", ">>"),
            ("«", "»"),
            ('"', '"'),
            ("“", "”"),
            ("‘", "’"),
            ("(", ")"),
            ("[", "]"),
            ("{", "}"),
        ]

        for left, right in pairs:
            if surface.startswith(left) and surface.endswith(right) and len(surface) > len(left) + len(right):
                raw = doc_text[start:end]
                lpos = raw.find(left)
                rpos = raw.rfind(right)
                if lpos != -1 and rpos != -1 and (rpos + len(right)) <= len(raw):
                    start += lpos + len(left)
                    end = start + (rpos - (lpos + len(left)))
                    changed = True
                    break

        while start < end and doc_text[start] in BOUNDARY_CHARS:
            start += 1
        while start < end and doc_text[end - 1] in BOUNDARY_CHARS:
            end -= 1

    if start >= end:
        return None

    surface = doc_text[start:end]
    m = PERSON_PREFIX_RE.match(surface)
    if m:
        start += len(m.group(0))

    while start < end and doc_text[start] in BOUNDARY_CHARS:
        start += 1
    while start < end and doc_text[end - 1] in BOUNDARY_CHARS:
        end -= 1

    if start >= end:
        return None

    norm_text = doc_text[start:end]

    if is_pgni_non_phi(norm_text):
        return None

    return {
        "start": int(start),
        "end": int(end),
        "label": label,
        "text": norm_text,
        "score": span.get("score")
    }


def normalize_spans_for_eval(spans, doc_text):
    out = []
    seen = set()

    for sp in spans:
        norm = normalize_eval_span(sp, doc_text)
        if norm is None:
            continue

        key = (norm["start"], norm["end"], norm["label"])
        if key not in seen:
            seen.add(key)
            out.append(norm)

    return sorted(out, key=lambda x: (x["start"], x["end"], x["label"]))


def is_mergeable_gap(gap_text):
    if gap_text is None:
        return False

    s = gap_text.strip()

    if s == "":
        return True

    return all(ch in BOUNDARY_CHARS for ch in s)


def merge_close_spans(spans, doc_text):
    if not spans:
        return []

    spans = sorted(spans, key=lambda x: (x["start"], x["end"], x["label"]))
    merged = [dict(spans[0])]

    for sp in spans[1:]:
        last = merged[-1]

        if sp["label"] != last["label"]:
            merged.append(dict(sp))
            continue

        if sp["start"] <= last["end"]:
            new_start = min(last["start"], sp["start"])
            new_end = max(last["end"], sp["end"])
            scores = [x for x in [last.get("score"), sp.get("score")] if x is not None]

            merged[-1] = {
                "start": new_start,
                "end": new_end,
                "label": last["label"],
                "text": doc_text[new_start:new_end],
                "score": float(np.mean(scores)) if scores else None
            }
            continue

        gap_text = doc_text[last["end"]:sp["start"]]

        if is_mergeable_gap(gap_text):
            new_start = last["start"]
            new_end = sp["end"]
            scores = [x for x in [last.get("score"), sp.get("score")] if x is not None]

            merged[-1] = {
                "start": new_start,
                "end": new_end,
                "label": last["label"],
                "text": doc_text[new_start:new_end],
                "score": float(np.mean(scores)) if scores else None
            }
        else:
            merged.append(dict(sp))

    return merged


# =========================================================
# Metrics
# =========================================================

def span_key(span):
    return (span["start"], span["end"], span["label"])


def spans_to_set(spans):
    return {span_key(s) for s in spans}


def overlaps(a, b):
    return not (a["end"] <= b["start"] or b["end"] <= a["start"])


def same_label(a, b):
    return a["label"] == b["label"]


def greedy_partial_match(pred_spans, gold_spans, require_same_label=True):
    matched_pred = set()
    matched_gold = set()
    matches = []

    for i, p in enumerate(pred_spans):
        for j, g in enumerate(gold_spans):
            if j in matched_gold:
                continue

            if require_same_label and not same_label(p, g):
                continue

            if overlaps(p, g):
                matched_pred.add(i)
                matched_gold.add(j)
                matches.append((i, j))
                break

    return matched_pred, matched_gold, matches


def compute_entity_metrics_for_doc(pred_spans, gold_spans, mode="strict"):
    pred_spans = sorted(pred_spans, key=lambda x: (x["start"], x["end"], x["label"]))
    gold_spans = sorted(gold_spans, key=lambda x: (x["start"], x["end"], x["label"]))

    if mode == "strict":
        pred_set = spans_to_set(pred_spans)
        gold_set = spans_to_set(gold_spans)

        tp = len(pred_set & gold_set)
        fp = len(pred_set - gold_set)
        fn = len(gold_set - pred_set)

    elif mode == "partial":
        matched_pred, matched_gold, _ = greedy_partial_match(
            pred_spans, gold_spans, require_same_label=True
        )
        tp = len(matched_pred)
        fp = len(pred_spans) - tp
        fn = len(gold_spans) - tp

    else:
        raise ValueError("mode must be 'strict' or 'partial'")

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


def aggregate_entity_metrics(per_doc_metrics):
    tp = sum(x["tp"] for x in per_doc_metrics)
    fp = sum(x["fp"] for x in per_doc_metrics)
    fn = sum(x["fn"] for x in per_doc_metrics)

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0.0

    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }


# =========================================================
# Evaluation
# =========================================================

def evaluate_entity_level_fold(root, exp_name, fold, positive_label="PHI", deduplicate=True):
    outputs = prepare_outputs(root, exp_name, fold)

    strict_doc_metrics = []
    partial_doc_metrics = []

    for doc_id in outputs["norm"]["eval_doc_ids"]:
        text = outputs["norm"]["text_by_doc_id"][doc_id]
        gold_raw = outputs["norm"]["entities_by_doc_id"][doc_id]

        gold_spans_raw = normalize_gold_entities(gold_raw, text, binary_phi=True)
        pred_spans_raw = get_doc_predicted_spans(
            outputs,
            doc_id=doc_id,
            deduplicate=deduplicate,
            positive_label=positive_label
        )

        gold_spans = normalize_spans_for_eval(gold_spans_raw, text)
        pred_spans = normalize_spans_for_eval(pred_spans_raw, text)

        gold_spans = merge_close_spans(gold_spans, text)
        pred_spans = merge_close_spans(pred_spans, text)

        strict_doc_metrics.append(
            compute_entity_metrics_for_doc(pred_spans, gold_spans, mode="strict")
        )
        partial_doc_metrics.append(
            compute_entity_metrics_for_doc(pred_spans, gold_spans, mode="partial")
        )

    strict = aggregate_entity_metrics(strict_doc_metrics)
    partial = aggregate_entity_metrics(partial_doc_metrics)
    m = outputs["metrics"]

    return {
        "experiment": exp_name,
        "fold": fold,

        "token_precision_phi": m.get("test_precision_phi"),
        "token_recall_phi": m.get("test_recall_phi"),
        "token_f1_phi": m.get("test_f1_phi"),

        "entity_strict_precision": strict["precision"],
        "entity_strict_recall": strict["recall"],
        "entity_strict_f1": strict["f1"],
        "entity_strict_tp": strict["tp"],
        "entity_strict_fp": strict["fp"],
        "entity_strict_fn": strict["fn"],

        "entity_partial_precision": partial["precision"],
        "entity_partial_recall": partial["recall"],
        "entity_partial_f1": partial["f1"],
        "entity_partial_tp": partial["tp"],
        "entity_partial_fp": partial["fp"],
        "entity_partial_fn": partial["fn"],

        "n_test_docs": m.get("n_test_docs"),
        "test_docs": str(m.get("test_docs")),
        "eval_doc_ids": str(outputs["norm"]["eval_doc_ids"]),
    }


def list_experiments(root):
    root = Path(root)
    return sorted([p.name for p in root.iterdir() if p.is_dir()])


def list_folds_for_experiment(root, exp_name):
    exp_dir = Path(root) / exp_name
    folds = []

    for p in exp_dir.glob("fold_*_metrics.json"):
        name = p.stem
        fold = int(name.split("_")[1])
        folds.append(fold)

    return sorted(set(folds))


def build_entity_results_df(root, positive_label="PHI", deduplicate=True):
    rows = []

    for exp_name in list_experiments(root):
        folds = list_folds_for_experiment(root, exp_name)

        for fold in folds:
            try:
                row = evaluate_entity_level_fold(
                    root=root,
                    exp_name=exp_name,
                    fold=fold,
                    positive_label=positive_label,
                    deduplicate=deduplicate
                )
                rows.append(row)
            except Exception as e:
                rows.append({
                    "experiment": exp_name,
                    "fold": fold,
                    "error": str(e)
                })

    return pd.DataFrame(rows).sort_values(["experiment", "fold"]).reset_index(drop=True)


# =========================================================
# Inspection
# =========================================================

def inspect_experiment_predictions(
    root,
    exp_name,
    fold,
    positive_label="PHI",
    deduplicate=True,
    max_docs=None,
    return_results=False,
    only_errors=False
):
    outputs = prepare_outputs(root, exp_name, fold)

    results = []

    for i, doc_id in enumerate(outputs["norm"]["eval_doc_ids"]):
        if max_docs is not None and i >= max_docs:
            break

        text = outputs["norm"]["text_by_doc_id"][doc_id]
        gold_raw = outputs["norm"]["entities_by_doc_id"][doc_id]

        gold_spans_raw = normalize_gold_entities(gold_raw, text, binary_phi=True)
        pred_spans_raw = get_doc_predicted_spans(
            outputs,
            doc_id=doc_id,
            deduplicate=deduplicate,
            positive_label=positive_label
        )

        gold_spans = normalize_spans_for_eval(gold_spans_raw, text)
        pred_spans = normalize_spans_for_eval(pred_spans_raw, text)

        gold_spans = merge_close_spans(gold_spans, text)
        pred_spans = merge_close_spans(pred_spans, text)

        gold_keys = {(g["start"], g["end"], g["text"]) for g in gold_spans}
        pred_keys = {(p["start"], p["end"], p["text"]) for p in pred_spans}

        if only_errors:
            gold_out = [
                {"start": g["start"], "end": g["end"], "text": g["text"]}
                for g in gold_spans
                if (g["start"], g["end"], g["text"]) not in pred_keys
            ]
            pred_out = [
                {"start": p["start"], "end": p["end"], "text": p["text"], "score": p.get("score")}
                for p in pred_spans
                if (p["start"], p["end"], p["text"]) not in gold_keys
            ]
        else:
            gold_out = [
                {"start": g["start"], "end": g["end"], "text": g["text"]}
                for g in gold_spans
            ]
            pred_out = [
                {"start": p["start"], "end": p["end"], "text": p["text"], "score": p.get("score")}
                for p in pred_spans
            ]

        if only_errors and not gold_out and not pred_out:
            continue

        results.append({
            "doc_id": doc_id,
            "gold": gold_out,
            "pred": pred_out,
        })

        print("\n" + "=" * 70)
        print(f"EXP: {exp_name} | FOLD: {fold} | DOC: {doc_id}")
        print("=" * 70)

        print("\nGOLD:")
        for g in gold_out:
            print(g)

        print("\nPRED:")
        for p in pred_out:
            print(p)

    if return_results:
        return results
    return None


def inspect_experiment_all_folds(
    root,
    exp_name,
    positive_label="PHI",
    deduplicate=True,
    max_docs_per_fold=None,
    return_results=False,
    only_errors=False
):
    folds = list_folds_for_experiment(root, exp_name)

    all_results = {}

    for fold in folds:
        print("\n" + "#" * 80)
        print(f"EXPERIMENT: {exp_name} | FOLD: {fold}")
        print("#" * 80)

        fold_results = inspect_experiment_predictions(
            root=root,
            exp_name=exp_name,
            fold=fold,
            positive_label=positive_label,
            deduplicate=deduplicate,
            max_docs=max_docs_per_fold,
            return_results=return_results,
            only_errors=only_errors
        )

        if return_results:
            all_results[fold] = fold_results

    if return_results:
        return all_results
    return None

In [ ]:
df = build_entity_results_df(ROOT)
df[[
    "experiment",
    "fold",
    "entity_strict_precision",
    "entity_strict_recall",
    "entity_strict_f1"
]]

,experiment,fold,entity_strict_precision,entity_strict_recall,entity_strict_f1
0,both_orig_orig,0,0.727273,0.727273,0.727273
1,both_orig_orig,1,0.750000,0.750000,0.750000
2,both_orig_orig,2,1.000000,0.909091,0.952381
3,both_orig_orig,3,1.000000,0.857143,0.923077
4,both_orig_orig,4,1.000000,1.000000,1.000000
5,both_syn_syn,0,0.762500,0.835616,0.797386
6,both_syn_syn,1,0.919540,0.851064,0.883978
7,both_syn_syn,2,0.983740,0.870504,0.923664
8,both_syn_syn,3,0.988889,0.978022,0.983425
9,both_syn_syn,4,0.989796,1.000000,0.994872


In [ ]:
summary_df = (
    df.groupby("experiment", as_index=False)
      .agg(
          strict_precision_mean=("entity_strict_precision", "mean"),
          strict_recall_mean=("entity_strict_recall", "mean"),
          strict_f1_mean=("entity_strict_f1", "mean"),
          n_folds=("fold", "count")
      )
      .sort_values("strict_f1_mean", ascending=False)
      .reset_index(drop=True)
)

summary_df

,experiment,strict_precision_mean,strict_recall_mean,strict_f1_mean,n_folds
0,syn_syn_syn,0.943083,0.941805,0.942260,5
1,both_syn_syn,0.928893,0.907041,0.916665,5
2,both_orig_orig,0.895455,0.848701,0.870546,5
3,orig_syn_syn,0.963705,0.573859,0.713352,5
4,orig_orig_orig,1.000000,0.603463,0.712698,5
5,syn_orig_orig,0.603333,0.701515,0.638235,5


In [ ]:
summary_df = (
    df.groupby("experiment", as_index=False)
      .agg(
          strict_precision_mean=("entity_strict_precision", "mean"),
          strict_precision_std=("entity_strict_precision", "std"),

          strict_recall_mean=("entity_strict_recall", "mean"),
          strict_recall_std=("entity_strict_recall", "std"),

          strict_f1_mean=("entity_strict_f1", "mean"),
          strict_f1_std=("entity_strict_f1", "std"),

          n_folds=("fold", "count")
      )
      .sort_values("strict_f1_mean", ascending=False)
      .reset_index(drop=True)
)

In [ ]:
summary_df["precision"] = summary_df.apply(
    lambda x: f"{x['strict_precision_mean']:.3f} ± {x['strict_precision_std']:.3f}", axis=1
)

summary_df["recall"] = summary_df.apply(
    lambda x: f"{x['strict_recall_mean']:.3f} ± {x['strict_recall_std']:.3f}", axis=1
)

summary_df["f1"] = summary_df.apply(
    lambda x: f"{x['strict_f1_mean']:.3f} ± {x['strict_f1_std']:.3f}", axis=1
)

In [ ]:
summary_df

,experiment,strict_precision_mean,strict_precision_std,strict_recall_mean,strict_recall_std,strict_f1_mean,strict_f1_std,n_folds,precision,recall,f1
0,syn_syn_syn,0.943083,0.072593,0.941805,0.055803,0.942260,0.063199,5,0.943 ± 0.073,0.942 ± 0.056,0.942 ± 0.063
1,both_syn_syn,0.928893,0.097585,0.907041,0.076239,0.916665,0.080496,5,0.929 ± 0.098,0.907 ± 0.076,0.917 ± 0.080
2,both_orig_orig,0.895455,0.143380,0.848701,0.113022,0.870546,0.123767,5,0.895 ± 0.143,0.849 ± 0.113,0.871 ± 0.124
3,orig_syn_syn,0.963705,0.046606,0.573859,0.135951,0.713352,0.116991,5,0.964 ± 0.047,0.574 ± 0.136,0.713 ± 0.117
4,orig_orig_orig,1.000000,0.000000,0.603463,0.305001,0.712698,0.267285,5,1.000 ± 0.000,0.603 ± 0.305,0.713 ± 0.267
5,syn_orig_orig,0.603333,0.324123,0.701515,0.372354,0.638235,0.341987,5,0.603 ± 0.324,0.702 ± 0.372,0.638 ± 0.342


In [ ]:
summary_df[["experiment", "precision", "recall", "f1"]].round(2)

,experiment,precision,recall,f1
0,syn_syn_syn,0.943 ± 0.073,0.942 ± 0.056,0.942 ± 0.063
1,both_syn_syn,0.929 ± 0.098,0.907 ± 0.076,0.917 ± 0.080
2,both_orig_orig,0.895 ± 0.143,0.849 ± 0.113,0.871 ± 0.124
3,orig_syn_syn,0.964 ± 0.047,0.574 ± 0.136,0.713 ± 0.117
4,orig_orig_orig,1.000 ± 0.000,0.603 ± 0.305,0.713 ± 0.267
5,syn_orig_orig,0.603 ± 0.324,0.702 ± 0.372,0.638 ± 0.342


In [ ]:
inspect_experiment_all_folds(
    root=ROOT,
    exp_name="orig_syn_syn",
    max_docs_per_fold=None,
    only_errors=False,
    return_results=False
)


################################################################################
EXPERIMENT: orig_syn_syn | FOLD: 0
################################################################################

EXP: orig_syn_syn | FOLD: 0 | DOC: 0

GOLD:
{'start': 114, 'end': 123, 'text': 'ΓΝ Δράμας'}
{'start': 457, 'end': 463, 'text': 'Δράμας'}

PRED:
{'start': 59, 'end': 75, 'text': 'ΤΕΠ λόγω σοβαρού', 'score': 0.9719239100813866}
{'start': 81, 'end': 92, 'text': 'ατος από αν', 'score': 0.9465461671352386}
{'start': 97, 'end': 123, 'text': 'μενο τροχαίο στο ΓΝ Δράμας', 'score': 0.9571126222610473}
{'start': 262, 'end': 272, 'text': 'ή, ο ασθεν', 'score': 0.9480931162834167}
{'start': 393, 'end': 408, 'text': 'τραύμα.\n\nΟ ασθε', 'score': 0.9731500670313835}
{'start': 454, 'end': 463, 'text': 'ΓΝ Δράμας', 'score': 0.961524498462677}
{'start': 500, 'end': 514, 'text': 'Πριν την εισαγ', 'score': 0.9541571617126465}
{'start': 662, 'end': 669, 'text': 'Εκτός α', 'score': 0.9329220553239187}
{'start':

In [ ]:
inspect_experiment_all_folds(
    root=ROOT,
    exp_name="syn_orig_orig",
    max_docs_per_fold=None,
    only_errors=True,
    return_results=False
)


################################################################################
EXPERIMENT: syn_orig_orig | FOLD: 0
################################################################################

################################################################################
EXPERIMENT: syn_orig_orig | FOLD: 1
################################################################################

EXP: syn_orig_orig | FOLD: 1 | DOC: 16

GOLD:
{'start': 2946, 'end': 2970, 'text': 'ΠΓΝ Βόλου «Αχιλλοπούλειο'}

PRED:
{'start': 2946, 'end': 2955, 'text': 'ΠΓΝ Βόλου', 'score': 0.9901225566864014}

EXP: syn_orig_orig | FOLD: 1 | DOC: 18

GOLD:
{'start': 3214, 'end': 3273, 'text': 'Αμαλία Μυτελέτσης Χριστοδούλα Ατσαλάκη Αντώνιος Σπασόπουλος'}

PRED:
{'start': 3221, 'end': 3273, 'text': 'Μυτελέτσης Χριστοδούλα Ατσαλάκη Αντώνιος Σπασόπουλος', 'score': 0.8982520198538191}

################################################################################
EXPERIMENT: syn_orig_orig | FOLD: 2
##########

In [ ]:
def prepare_outputs(root, exp_name, fold):
    outputs = load_fold_outputs(root, exp_name, fold)

    metrics_test_docs = normalize_test_docs(outputs["metrics"])
    eval_doc_ids = get_eval_doc_ids(outputs)

    meta = outputs["meta"]

    if "test_instance_texts" in meta:
        raw_texts = meta["test_instance_texts"]
    elif "test_doc_texts" in meta:
        raw_texts = meta["test_doc_texts"]
    else:
        raise KeyError(f"No text field found in meta. Available keys: {list(meta.keys())}")

    if "test_instance_entities" in meta:
        raw_entities = meta["test_instance_entities"]
    elif "test_doc_entities" in meta:
        raw_entities = meta["test_doc_entities"]
    else:
        raise KeyError(f"No entity field found in meta. Available keys: {list(meta.keys())}")

    text_by_doc_id = normalize_items_by_doc_id(
        raw=raw_texts,
        eval_doc_ids=eval_doc_ids,
        metrics_test_docs=metrics_test_docs,
        field_name="texts",
    )

    entities_by_doc_id = normalize_items_by_doc_id(
        raw=raw_entities,
        eval_doc_ids=eval_doc_ids,
        metrics_test_docs=metrics_test_docs,
        field_name="entities",
    )

    outputs["norm"] = {
        "metrics_test_docs": metrics_test_docs,
        "eval_doc_ids": eval_doc_ids,
        "text_by_doc_id": text_by_doc_id,
        "entities_by_doc_id": entities_by_doc_id,
    }

    return outputs

In [ ]:
def get_doc_predicted_spans(outputs, doc_id, deduplicate=True, positive_label="PHI"):
    meta = outputs["meta"]
    logits_all = outputs["logits"]

    text_by_doc_id = outputs["norm"]["text_by_doc_id"]
    full_text = text_by_doc_id[int(doc_id)]

    offset_mapping = meta["offset_mapping"]
    texts_per_window = meta.get("texts_per_window", None)

    abs_spans = []
    search_from = 0

    for widx in get_doc_window_indices(outputs, doc_id):
        local_offsets = normalize_offsets(offset_mapping[widx])

        local_logits = ensure_numeric_2d(logits_all[widx])
        local_preds = local_logits.argmax(axis=-1)
        local_probs = softmax(local_logits, axis=-1)

        local_spans = bio_tokens_to_local_spans(
            pred_ids=local_preds,
            offsets=local_offsets,
            probs=local_probs,
            positive_label=positive_label
        )

        # CASE 1: old/original path with window text available
        if texts_per_window is not None:
            window_text = texts_per_window[widx]
            window_start = locate_window_in_doc(full_text, window_text, search_from=search_from)
            if window_start == -1:
                continue

            search_from = max(search_from, window_start)

            for sp in local_spans:
                abs_start = window_start + sp["start"]
                abs_end = window_start + sp["end"]

                if 0 <= abs_start < abs_end <= len(full_text):
                    abs_spans.append({
                        "start": abs_start,
                        "end": abs_end,
                        "label": sp["label"],
                        "text": full_text[abs_start:abs_end],
                        "score": sp["score"]
                    })

        # CASE 2: synthetic path, assume offsets are already absolute in full doc
        else:
            for sp in local_spans:
                abs_start = sp["start"]
                abs_end = sp["end"]

                if 0 <= abs_start < abs_end <= len(full_text):
                    abs_spans.append({
                        "start": abs_start,
                        "end": abs_end,
                        "label": sp["label"],
                        "text": full_text[abs_start:abs_end],
                        "score": sp["score"]
                    })

    if deduplicate:
        abs_spans = deduplicate_spans(abs_spans)

    return abs_spans

In [ ]:
_ = inspect_experiment_predictions(
    root=ROOT,
    exp_name="orig_syn_syn",
    fold=0,
    max_docs=1,
    only_errors=False,
    return_results=False
)


EXP: orig_syn_syn | FOLD: 0 | DOC: 0

GOLD:
{'start': 114, 'end': 123, 'text': 'ΓΝ Δράμας'}
{'start': 457, 'end': 463, 'text': 'Δράμας'}

PRED:
{'start': 59, 'end': 75, 'text': 'ΤΕΠ λόγω σοβαρού', 'score': 0.9719239100813866}
{'start': 81, 'end': 92, 'text': 'ατος από αν', 'score': 0.9465461671352386}
{'start': 97, 'end': 123, 'text': 'μενο τροχαίο στο ΓΝ Δράμας', 'score': 0.9571126222610473}
{'start': 262, 'end': 272, 'text': 'ή, ο ασθεν', 'score': 0.9480931162834167}
{'start': 393, 'end': 408, 'text': 'τραύμα.\n\nΟ ασθε', 'score': 0.9731500670313835}
{'start': 454, 'end': 463, 'text': 'ΓΝ Δράμας', 'score': 0.961524498462677}
{'start': 500, 'end': 514, 'text': 'Πριν την εισαγ', 'score': 0.9541571617126465}
{'start': 662, 'end': 669, 'text': 'Εκτός α', 'score': 0.9329220553239187}
{'start': 1208, 'end': 1217, 'text': 'τακτικά γ', 'score': 0.95018470287323}
{'start': 1276, 'end': 1286, 'text': 'α διατηρεί', 'score': 0.8411791125933329}


In [ ]:
import html
from IPython.display import display, HTML
import ipywidgets as widgets

In [ ]:
def get_by_doc_id(d, doc_id):
    if doc_id in d:
        return d[doc_id]

    doc_id_int = int(doc_id)
    if doc_id_int in d:
        return d[doc_id_int]

    doc_id_str = str(doc_id_int)
    if doc_id_str in d:
        return d[doc_id_str]

    # useful debug
    raise KeyError(
        f"doc_id={doc_id} not found. "
        f"Sample available keys: {list(d.keys())[:10]}"
    )

def get_gold_pred_for_doc(root, exp_name, fold, doc_id, positive_label="PHI", deduplicate=True):
    outputs = prepare_outputs(root, exp_name, fold)

    text = get_by_doc_id(outputs["norm"]["text_by_doc_id"], doc_id)
    gold_raw = get_by_doc_id(outputs["norm"]["entities_by_doc_id"], doc_id)

    gold_spans_raw = normalize_gold_entities(gold_raw, text, binary_phi=True)
    pred_spans_raw = get_doc_predicted_spans(
        outputs,
        doc_id=int(doc_id),
        deduplicate=deduplicate,
        positive_label=positive_label
    )

    gold_spans = normalize_spans_for_eval(gold_spans_raw, text)
    pred_spans = normalize_spans_for_eval(pred_spans_raw, text)

    gold_spans = merge_close_spans(gold_spans, text)
    pred_spans = merge_close_spans(pred_spans, text)

    return text, gold_spans, pred_spans

In [ ]:
def build_highlighted_html(text, gold_spans, pred_spans):
    events = []

    for sp in gold_spans:
        events.append((sp["start"], "gold_start"))
        events.append((sp["end"], "gold_end"))

    for sp in pred_spans:
        events.append((sp["start"], "pred_start"))
        events.append((sp["end"], "pred_end"))

    priority = {
        "gold_end": 0,
        "pred_end": 1,
        "gold_start": 2,
        "pred_start": 3,
    }
    events.sort(key=lambda x: (x[0], priority[x[1]]))

    html_parts = []
    gold_open = 0
    pred_open = 0
    pos = 0

    def current_style(gold_on, pred_on):
        if gold_on and pred_on:
            return "background:rgba(76, 175, 80, 0.35); border-bottom:2px solid #66bb6a;"
        elif gold_on:
            return "background:rgba(33, 150, 243, 0.35); border-bottom:2px solid #42a5f5;"
        elif pred_on:
            return "background:rgba(244, 67, 54, 0.35); border-bottom:2px solid #ef5350;"
        return ""

    def wrap_segment(seg, gold_on, pred_on):
        esc = html.escape(seg)
        style = current_style(gold_on, pred_on)
        if style:
            return f'<span style="{style}">{esc}</span>'
        return esc

    for ev_pos, ev_type in events:
        if ev_pos > pos:
            html_parts.append(wrap_segment(text[pos:ev_pos], gold_open > 0, pred_open > 0))
            pos = ev_pos

        if ev_type == "gold_end":
            gold_open -= 1
        elif ev_type == "pred_end":
            pred_open -= 1
        elif ev_type == "gold_start":
            gold_open += 1
        elif ev_type == "pred_start":
            pred_open += 1

    if pos < len(text):
        html_parts.append(wrap_segment(text[pos:], gold_open > 0, pred_open > 0))

    legend = """
    <div style="margin-bottom:10px; color:#ddd;">
      <span style="background:rgba(76,175,80,0.35); padding:2px 6px; border-radius:4px;">✓ correct</span>
      <span style="background:rgba(33,150,243,0.35); padding:2px 6px; border-radius:4px; margin-left:8px;">gold only</span>
      <span style="background:rgba(244,67,54,0.35); padding:2px 6px; border-radius:4px; margin-left:8px;">pred only</span>
    </div>
    """

    content = "".join(html_parts).replace("\n", "<br>")

    box = f"""
    <div style="
        background:#1e1e1e;
        color:#e0e0e0;
        white-space:normal;
        line-height:1.7;
        font-family:monospace;
        font-size:14px;
        border:1px solid #444;
        padding:12px;
        border-radius:8px;
        max-height:600px;
        overflow:auto;
    ">
        {content}
    </div>
    """

    return legend + box

In [ ]:
def format_spans_html(spans, show_score=False):
    rows = []
    for sp in spans:
        if show_score:
            score = sp.get("score")
            rows.append(
                f"<li><b>{sp['start']}-{sp['end']}</b> | "
                f"{html.escape(sp['text'])} | "
                f"score={'' if score is None else round(float(score), 4)}</li>"
            )
        else:
            rows.append(
                f"<li><b>{sp['start']}-{sp['end']}</b> | {html.escape(sp['text'])}</li>"
            )

    if not rows:
        return "<i>None</i>"

    return "<ul>" + "".join(rows) + "</ul>"

In [ ]:
def interactive_experiment_viewer(root, positive_label="PHI", deduplicate=True):
    experiments = list_experiments(root)

    exp_dropdown = widgets.Dropdown(
        options=experiments,
        description="Experiment:",
        layout=widgets.Layout(width="420px")
    )

    fold_dropdown = widgets.Dropdown(
        options=[],
        description="Fold:",
        layout=widgets.Layout(width="180px")
    )

    doc_dropdown = widgets.Dropdown(
        options=[],
        description="Doc:",
        layout=widgets.Layout(width="220px")
    )

    out = widgets.Output()
    cache = {}
    updating = {"busy": False}

    def get_outputs(exp_name, fold):
        key = (exp_name, int(fold))
        if key not in cache:
            cache[key] = prepare_outputs(root, exp_name, int(fold))
        return cache[key]

    def get_valid_doc_ids(exp_name, fold):
        outputs = get_outputs(exp_name, fold)
        return sorted(int(k) for k in outputs["norm"]["text_by_doc_id"].keys())

    def refresh_folds(*args):
        updating["busy"] = True
        try:
            exp_name = exp_dropdown.value
            folds = list_folds_for_experiment(root, exp_name)
            fold_dropdown.options = folds
            if folds:
                fold_dropdown.value = folds[0]
        finally:
            updating["busy"] = False

    def refresh_docs(*args):
        exp_name = exp_dropdown.value
        fold = fold_dropdown.value
        if exp_name is None or fold is None:
            return

        updating["busy"] = True
        try:
            doc_ids = get_valid_doc_ids(exp_name, fold)
            doc_dropdown.options = doc_ids

            if not doc_ids:
                doc_dropdown.value = None
                return

            if doc_dropdown.value not in doc_ids:
                doc_dropdown.value = doc_ids[0]
        finally:
            updating["busy"] = False

    def render(*args):
        if updating["busy"]:
            return

        out.clear_output()

        exp_name = exp_dropdown.value
        fold = fold_dropdown.value
        doc_id = doc_dropdown.value

        if exp_name is None or fold is None:
            return

        valid_doc_ids = get_valid_doc_ids(exp_name, fold)
        if not valid_doc_ids:
            with out:
                display(HTML("<b>No docs available.</b>"))
            return

        if doc_id not in valid_doc_ids:
            updating["busy"] = True
            try:
                doc_dropdown.value = valid_doc_ids[0]
            finally:
                updating["busy"] = False
            return

        text, gold_spans, pred_spans = get_gold_pred_for_doc(
            root=root,
            exp_name=exp_name,
            fold=int(fold),
            doc_id=int(doc_id),
            positive_label=positive_label,
            deduplicate=deduplicate
        )

        html_text = build_highlighted_html(text, gold_spans, pred_spans)
        gold_html = format_spans_html(gold_spans, show_score=False)
        pred_html = format_spans_html(pred_spans, show_score=True)

        panel_html = f"""
        <div style="display:grid; grid-template-columns: 2fr 1fr; gap:16px; align-items:start;">
            <div>
                <h3 style="margin:0 0 10px 0; color:#ddd;">Text</h3>
                {html_text}
            </div>
            <div style="color:#ddd;">
                <h3 style="margin:0 0 8px 0;">Gold</h3>
                {gold_html}
                <h3 style="margin:16px 0 8px 0;">Pred</h3>
                {pred_html}
            </div>
        </div>
        """

        with out:
            display(HTML(panel_html))

    exp_dropdown.observe(refresh_folds, names="value")
    exp_dropdown.observe(refresh_docs, names="value")
    exp_dropdown.observe(render, names="value")

    fold_dropdown.observe(refresh_docs, names="value")
    fold_dropdown.observe(render, names="value")

    doc_dropdown.observe(render, names="value")

    refresh_folds()
    refresh_docs()
    render()

    controls = widgets.HBox([exp_dropdown, fold_dropdown, doc_dropdown])
    display(widgets.VBox([controls, out]))

In [ ]:
interactive_experiment_viewer(ROOT)

recall 1

In [ ]:
def evaluate_entity_level_fold_with_threshold(
    root,
    exp_name,
    fold,
    threshold=0.0,
    positive_label="PHI",
    deduplicate=True,
):
    outputs = prepare_outputs(root, exp_name, fold)

    strict_doc_metrics = []

    for doc_id in outputs["norm"]["eval_doc_ids"]:
        text = outputs["norm"]["text_by_doc_id"][doc_id]
        gold_raw = outputs["norm"]["entities_by_doc_id"][doc_id]

        gold_spans_raw = normalize_gold_entities(gold_raw, text, binary_phi=True)

        pred_spans_raw = get_doc_predicted_spans(
            outputs,
            doc_id=doc_id,
            deduplicate=deduplicate,
            positive_label=positive_label
        )

        pred_spans_raw = [
            sp for sp in pred_spans_raw
            if sp.get("score") is None or sp["score"] >= threshold
        ]

        gold_spans = normalize_spans_for_eval(gold_spans_raw, text)
        pred_spans = normalize_spans_for_eval(pred_spans_raw, text)

        gold_spans = merge_close_spans(gold_spans, text)
        pred_spans = merge_close_spans(pred_spans, text)

        strict_doc_metrics.append(
            compute_entity_metrics_for_doc(pred_spans, gold_spans, mode="strict")
        )

    strict = aggregate_entity_metrics(strict_doc_metrics)

    return {
        "threshold": threshold,
        "precision": strict["precision"],
        "recall": strict["recall"],
        "f1": strict["f1"],
        "tp": strict["tp"],
        "fp": strict["fp"],
        "fn": strict["fn"],
    }

In [ ]:
def sweep_thresholds(
    root,
    exp_name,
    fold,
    thresholds=None,
):
    if thresholds is None:
        thresholds = np.linspace(0.0, 1.0, 101)

    rows = []

    for t in thresholds:
        m = evaluate_entity_level_fold_with_threshold(
            root=root,
            exp_name=exp_name,
            fold=fold,
            threshold=float(t),
        )
        rows.append(m)

    return pd.DataFrame(rows)

In [ ]:
def get_precision_at_recall_one(df, recall_tol=0.999):
    df_valid = df[df["recall"] >= recall_tol]

    if len(df_valid) == 0:
        return None

    best = df_valid.sort_values("precision", ascending=False).iloc[0]
    return best

In [ ]:
def get_precision_at_target_recall(df, target_recall=1.0, tol=1e-9):
    df = df.sort_values(["recall", "precision"], ascending=[False, False]).reset_index(drop=True)

    eligible = df[df["recall"] >= (target_recall - tol)]
    if len(eligible) > 0:
        best = eligible.sort_values("precision", ascending=False).iloc[0]
        return {
            "found": True,
            "target_recall": target_recall,
            "best_row": best,
            "max_recall_in_sweep": df["recall"].max(),
        }

    best_max_recall = df[df["recall"] == df["recall"].max()].sort_values("precision", ascending=False).iloc[0]
    return {
        "found": False,
        "target_recall": target_recall,
        "best_row": best_max_recall,
        "max_recall_in_sweep": df["recall"].max(),
    }

In [ ]:
df = sweep_thresholds(ROOT, "both_orig_orig", fold=2)

res = get_precision_at_target_recall(df, target_recall=1.0)
print(res["found"])
print("max recall in sweep:", res["max_recall_in_sweep"])
print(res["best_row"])

False
max recall in sweep: 0.9090909090909091
threshold     0.000000
precision     1.000000
recall        0.909091
f1            0.952381
tp           10.000000
fp            0.000000
fn            1.000000
Name: 0, dtype: float64
